# CloudVision — GitHub ↔ Kaggle bridge

شغّل هذه الخلية في جلسة Kaggle الحالية بعد اختيار GPU T4. الجسر ينتظر أوامر محددة ومسموحاً بها فقط من فرع GitHub التجريبي، وينشر تقارير التنفيذ إلى المستودع. لا تضع أي مفتاح داخل نص الدفتر.

In [ ]:
import os, sys, subprocess, requests
from kaggle_secrets import UserSecretsClient

# خزّن GitHub fine-grained PAT في Kaggle Secrets باسم GITHUB_TOKEN.
# يجب أن يملك صلاحية Contents: Read and write لهذا المستودع فقط.
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not token:
    raise RuntimeError('أضف GITHUB_TOKEN إلى Kaggle Secrets أولاً؛ لا ترسله في المحادثة.')
os.environ['GITHUB_TOKEN'] = token

url = 'https://raw.githubusercontent.com/khd98me-svg/TermuxAgent/cloudvision-kaggle-bridge/cloudvision/kaggle_bridge.py'
response = requests.get(url, timeout=30)
response.raise_for_status()
script_path = '/kaggle/working/kaggle_bridge.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(response.text)
print('Bridge script downloaded. Starting listener...')
print('Keep this cell running; interrupt it to stop the bridge.')
subprocess.run([sys.executable, script_path], check=True)

## الأمان والتشغيل
- لا ترسل Cookies أو Session IDs أو مفاتيح Kaggle أو GitHub في المحادثة.
- أنشئ GitHub fine-grained token بصلاحية Contents Read/Write لهذا المستودع فقط، ثم خزّنه في Kaggle Secrets بالاسم `GITHUB_TOKEN`.
- الجسر لا يقبل أوامر shell عامة؛ يسمح فقط بـ `cuda_info` و `realesrgan_image_test`.
- تبقى الجلسة خاضعة لحدود Kaggle وقد تنتهي تلقائياً؛ هذا ليس خادماً دائماً.
- التقارير تظهر في `cloudvision/bridge-results/` على الفرع التجريبي.